# LFM North-Polar WAC + Static Tiling Example

This notebook demonstrates the easy, configuration-driven lunar tiling workflow. A user supplies raster directories, a geographic point or AOI, and an optional product ID. The workflow then prepares or reuses each raster index, selects the appropriate numbered LTM or polar grid, chooses modality-appropriate zoom levels, and writes structured lunar-grid cubes.

This copy retains the two main WAC and NAC examples and adds an enabled north-polar experiment that combines WAC with three staged LOLA/Kaguya static VRTs. The query supplies only an AOI and optional WAC product ID; routing selects `LPS_N` automatically. If a product does not intersect an otherwise valid query, the backend warns, skips that product, and lets other matching dynamic sources continue; when none match, it returns no records and the notebook skips the plot. All tiling warps use bilinear resampling, and results are returned as `TileCubeRecord` objects so downstream code does not need to parse filenames.

### Setup

Select `lfm_kernel` from the kernel menu in the top-right corner before running this notebook.

## Imports and repository discovery

Run this notebook from the repository's top-level `notebooks/` directory. JupyterHub may expose the clone through `/panfs`; repository discovery normalizes that path to the equivalent `/explore` symlink before importing LFM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists() or not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find the lfm/ and model/ directories. Run this notebook "
        "from the repository's top-level notebooks/ directory."
    )

sys.path.insert(0, str(repo_root))

from model import (
    TileAOIQuery,
    TileConfig,
    TilePointQuery,
    TileSourceConfig,
    TileSourceDefinition,
    create_tiles_for_index,
    create_tiles_for_query,
    default_zoom_for_modality,
    make_nac_tile_source,
    make_static_tile_source,
    make_wac_tile_source,
    resolve_notebook_source_index,
)
from lfm.all_models.all_tasks.tiling_utils import (
    DEFAULT_NAC_BAND_NUMBER,
    DEFAULT_WAC_BAND_NUMBER,
    RUN_ID,
)
from lfm.all_models.all_tasks.viz import (
    plot_tiling_records,
    print_record_summary,
)

print(f"Repository root: {repo_root}")
print("Successfully imported the automatic LFM tiling workflow")

## User configuration

These are the values a user normally edits. The defaults reference representative Explore collections used by the tiling validation suite.

### Data directories

- `WAC_DATA_DIR` and `NAC_DATA_DIR` contain the main processed image collections.
- `STATIC_DATA_DIR` contains the canonical 63-band static context collection.
- `POLAR_WAC_DATA_DIR` contains the global polar WAC VRT used only by the opt-in alternate example.
- `POLAR_STATIC_DATA_DIR` is a user-owned staging directory containing copied north-polar static VRTs and symlinks to their backing TIFFs.

The default WAC, NAC, and static directories use their existing shared `output_index.gpkg` files read-only. If you replace any data directory with a custom collection, the notebook instead creates a user-owned GeoPackage under this clone's persistent index cache. The first custom-directory run scans `.tif`, `.tiff`, `.nc`, and `.vrt` rasters and can take several minutes; later runs validate and reuse that cache. Progress and reuse messages are printed to notebook stdout.

### Products and geographic AOIs

- Set a product ID to an exact filename-derived product prefix to request one observation. Set it to `None` to discover every product intersecting that AOI. Companion files belonging to one product are grouped, while unrelated products are written to separate cubes.
- AOI dictionaries use lunar geographic IAU:30100 coordinates in the order `ul_lat`, `ul_lon`, `lr_lat`, `lr_lon`. They describe a search region, not a clipping boundary: tiling writes every complete 512×512 tile selected by the AOI, so output footprints can extend beyond it.

### Source inclusion

- `INCLUDE_DYNAMIC=True` and `INCLUDE_STATIC=True` is the default dynamic-plus-static workflow.
- Disable static for imagery-only cubes, or disable dynamic for static-only cubes. At least one must remain enabled. Disabled source collections are not indexed, validated, queried, or written.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")  # Shared project-data root on Explore.
WAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_WAC_Pho_Sites"  # Processed WAC rasters.
NAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_NAC_Pho_Sites"  # Processed NAC rasters.
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"  # Canonical static lunar rasters.
POLAR_WAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/Static_final/LROC/WAC/wac_glob_morf_mos"  # Polar WAC VRTs.
POLAR_STATIC_DATA_DIR = Path("/explore/nobackup/people/ajkerr1/lfm_polar_static/north_60m")  # Staged static VRTs plus backing-TIFF symlinks.
POLAR_STATIC_IMAGE_GLOBS = (
    "lola_kaguya_60mpp.60n.ste.asp.LPS_N.vrt",
    "lola_kaguya_60mpp.60n.ste.cos.LPS_N.vrt",
    "lola_kaguya_60mpp.60n.ste.elv.LPS_N.vrt",
)

WAC_PRODUCT_ID = "M1107459759CE"  # Exact WAC prefix, or None to discover intersecting PIDs.
NAC_PRODUCT_ID = "M1117899885LE"  # Exact NAC prefix, or None to discover intersecting PIDs.
POLAR_WAC_PRODUCT_ID = None  # Discover the product intersecting the polar AOI.

# IAU:30100 latitude/longitude bounds intersecting WAC_PRODUCT_ID.
WAC_AOI_BOUNDS = {
    "ul_lat": 31.241501479374616,  # Northern edge.
    "ul_lon": 121.01616204071722,  # Western edge.
    "lr_lat": 30.201163639632075,  # Southern edge.
    "lr_lon": 122.22643208573085,  # Eastern edge.
}

# IAU:30100 latitude/longitude bounds centered inside NAC_PRODUCT_ID.
NAC_AOI_BOUNDS = {
    "ul_lat": 1.0786543156953,  # Northern edge.
    "ul_lon": 149.752054273755,  # Western edge.
    "lr_lat": 1.0586543156953,  # Southern edge.
    "lr_lon": 149.772054273755,  # Eastern edge.
}

# North-polar AOI used by the enabled WAC + staged-static experiment.
POLAR_WAC_AOI_BOUNDS = {
    "ul_lat": 86.1,
    "ul_lon": -0.1,
    "lr_lat": 85.9,
    "lr_lon": 0.1,
}

INCLUDE_DYNAMIC = True  # Write WAC/NAC cubes in the two main examples.
INCLUDE_STATIC = True  # Write canonical static context beside main dynamic cubes.

## Set up derived values

Most users can leave this section unchanged. It selects shared indexes or per-clone custom-data caches, derives modality-specific default zooms, creates run directories, and sets display controls. WAC defaults to LTM zoom 5 and polar zoom 4; processed one-metre NAC defaults to LTM zoom 11 and polar zoom 10. These are near-native choices, and the automatic workflow applies the correct family value after routing. Expert callers can pass `zoom_overrides` when constructing a source.

`STATIC_DATA_DIR` is configured to the standard 63-layer collection. The tiling code can use a different static collection, but it must be configured with its own layers, missing-data rules, display layer, and index. **If you change the variable value at the moment the notebook will no longer work properly.**

For each default data directory, the resolver requires and reuses that directory's shared `output_index.gpkg`; shared indexes are never automatically replaced. A changed data directory receives a persistent, user-owned cache under `INDEX_CACHE_DIR`, where an invalid or stale GeoPackage can be rebuilt safely. The staged north-polar static collection has its own persistent cache and explicit three-VRT inventory. Missing custom-index creation uses `SLURM_CPUS_PER_TASK` footprint workers by default; set `INDEX_WORKER_COUNT=1` to force serial indexing. The timestamped output directory prevents accidental cube reuse. `MAX_PLOT_TILES` limits only the number of displayed tile/product samples; every selected cube is still written. `RUN_POLAR_STATIC_QUERY=True` runs the WAC + staged-static polar example; `RUN_ALTERNATE_QUERIES` independently controls the point and explicit-index examples.

In [ ]:
if not INCLUDE_DYNAMIC and not INCLUDE_STATIC:
    raise ValueError("At least one of INCLUDE_DYNAMIC or INCLUDE_STATIC must be True.")

BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "tiling"  # Notebook-local parent for caches and runs.
INDEX_CACHE_DIR = BASE_OUTPUT_DIR / "indexes"  # Persistent, user-owned source-index cache.
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_ID  # Unique timestamped directory for this run.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)  # Refuse to reuse an existing run directory.

LOCATION_FIELD = "location"  # Vector-index field containing each raster path.
INDEX_WORKER_COUNT = None  # None uses SLURM_CPUS_PER_TASK; 1 forces serial indexing.
WAC_INDEX_RESOLUTION = resolve_notebook_source_index(
    source_name="wac", data_dir=WAC_DATA_DIR, cache_dir=INDEX_CACHE_DIR
) if INCLUDE_DYNAMIC else None
NAC_INDEX_RESOLUTION = resolve_notebook_source_index(
    source_name="nac", data_dir=NAC_DATA_DIR, cache_dir=INDEX_CACHE_DIR
) if INCLUDE_DYNAMIC else None
STATIC_INDEX_RESOLUTION = resolve_notebook_source_index(
    source_name="static", data_dir=STATIC_DATA_DIR, cache_dir=INDEX_CACHE_DIR
) if INCLUDE_STATIC else None
WAC_INDEX = WAC_INDEX_RESOLUTION.index_path if WAC_INDEX_RESOLUTION else None
NAC_INDEX = NAC_INDEX_RESOLUTION.index_path if NAC_INDEX_RESOLUTION else None
STATIC_INDEX = STATIC_INDEX_RESOLUTION.index_path if STATIC_INDEX_RESOLUTION else None
POLAR_WAC_INDEX = OUTPUT_DIR / "indexes" / "polar_wac_index.gpkg"
POLAR_STATIC_INDEX = INDEX_CACHE_DIR / "polar_static_north_60m_index.gpkg"

WAC_LTM_ZOOM = default_zoom_for_modality("wac", "ltm")  # Approximately 75.8 m/pixel.
NAC_LTM_ZOOM = default_zoom_for_modality("nac", "ltm")  # Approximately 1.185 m/pixel.
POLAR_STATIC_ZOOM = 4  # Approximately 73.8 m/pixel for the staged 60 m products.

WAC_BAND_NUMBER = DEFAULT_WAC_BAND_NUMBER  # First WAC VIS channel in the main collection.
NAC_BAND_NUMBER = DEFAULT_NAC_BAND_NUMBER  # NAC single channel.
POLAR_WAC_BAND_NUMBER = 1  # The example polar WAC VRT contains one band.
STATIC_BAND_TO_PLOT = "lola_kaguya_60mpp_elv"  # Static elevation band shown in figures.
MAX_PLOT_TILES = 4  # Display-only limit; it does not limit cube creation.
RUN_ALTERNATE_QUERIES = False  # True runs point and explicit-index examples.
RUN_POLAR_STATIC_QUERY = True  # Run the dedicated north-polar WAC + static example.

print(f"Notebook outputs will default here: {OUTPUT_DIR}")
for resolution in (WAC_INDEX_RESOLUTION, NAC_INDEX_RESOLUTION, STATIC_INDEX_RESOLUTION):
    if resolution is not None:
        ownership = "shared default" if resolution.uses_shared_default else "per-clone cache"
        print(f"{resolution.source_name.upper()} index ({ownership}): {resolution.index_path}")

## Resolve data sources

These constructors describe source policy without building an index. The first call to `create_tiles_for_query()` validates each enabled index. Existing shared default GeoPackages are reused read-only. For a custom data directory, a missing per-clone cache is created and an invalid or stale cache is rebuilt with visible per-raster progress. The tiler itself continues to query prepared indexes read-only and never modifies shared rasters or indexes.

WAC and NAC are product-scoped and preserve native source NoData. The static source below uses the repository's standard 63-layer preset, including its band order, bilinear resampling, and `-32768` output NoData. To use a different static collection, replace this preset with a configuration describing that collection; changing only `STATIC_DATA_DIR` is not sufficient. Separate WAC- and NAC-context definitions provide an unambiguous zoom if the notebook is switched to static-only mode.

In [ ]:
WAC_SOURCE = (
    make_wac_tile_source(
        data_dir=WAC_DATA_DIR,
        index_path=WAC_INDEX,
        location_field=LOCATION_FIELD,
        rebuild_invalid_index=WAC_INDEX_RESOLUTION.rebuild_invalid_index,
        index_worker_count=INDEX_WORKER_COUNT,
    )
    if INCLUDE_DYNAMIC
    else None
)
NAC_SOURCE = (
    make_nac_tile_source(
        data_dir=NAC_DATA_DIR,
        index_path=NAC_INDEX,
        location_field=LOCATION_FIELD,
        required=False,  # Sparse NAC coverage may legitimately produce no record.
        rebuild_invalid_index=NAC_INDEX_RESOLUTION.rebuild_invalid_index,
        index_worker_count=INDEX_WORKER_COUNT,
    )
    if INCLUDE_DYNAMIC
    else None
)
WAC_STATIC_SOURCE = (
    make_static_tile_source(
        data_dir=STATIC_DATA_DIR,
        index_path=STATIC_INDEX,
        location_field=LOCATION_FIELD,
        zoom_overrides={"ltm": WAC_LTM_ZOOM},
        rebuild_invalid_index=STATIC_INDEX_RESOLUTION.rebuild_invalid_index,
        index_worker_count=INDEX_WORKER_COUNT,
    )
    if INCLUDE_STATIC
    else None
)
NAC_STATIC_SOURCE = (
    make_static_tile_source(
        data_dir=STATIC_DATA_DIR,
        index_path=STATIC_INDEX,
        location_field=LOCATION_FIELD,
        zoom_overrides={"ltm": NAC_LTM_ZOOM},
        rebuild_invalid_index=STATIC_INDEX_RESOLUTION.rebuild_invalid_index,
        index_worker_count=INDEX_WORKER_COUNT,
    )
    if INCLUDE_STATIC
    else None
)

## Example 1: WAC with optional static context

The geographic AOI is routed automatically. Because this example is below 82° latitude, it uses numbered LTM grids and the WAC LTM default zoom. `WAC_PRODUCT_ID=None` discovers all intersecting observations and writes each product separately. Static context is written once per tile, even when multiple WAC products are found.

In [ ]:
wac_records = create_tiles_for_query(
    query=TileAOIQuery(**WAC_AOI_BOUNDS),
    output_dir=OUTPUT_DIR / "wac",
    dynamic_sources=((WAC_SOURCE,) if WAC_SOURCE is not None else ()),
    static_sources=((WAC_STATIC_SOURCE,) if WAC_STATIC_SOURCE is not None else ()),
    product_ids=({"wac": WAC_PRODUCT_ID} if INCLUDE_DYNAMIC else None),
    include_dynamic=INCLUDE_DYNAMIC,
    include_static=INCLUDE_STATIC,
    stdout=sys.stdout,  # Show index creation progress or reuse messages.
)
print_record_summary(wac_records)
resolved_wac_product_ids = sorted({
    record.product_id
    for record in wac_records
    if record.source_name == "wac" and record.product_id is not None
})
print(f"Resolved WAC product IDs: {resolved_wac_product_ids}")

if wac_records:
    wac_figure = plot_tiling_records(
        wac_records,
        dynamic_source="wac",
        dynamic_label="WAC",
        dynamic_band_number=WAC_BAND_NUMBER,
        static_band_name=STATIC_BAND_TO_PLOT,
        output_path=OUTPUT_DIR / "plots" / "wac_cubes.png",
        max_tiles=MAX_PLOT_TILES,
    )
else:
    wac_figure = None
    print("No WAC or static records were produced; skipping the WAC plot.")

## Example 2: NAC with optional static context

The same workflow uses the processed-NAC defaults automatically. Below the polar threshold, NAC is written at LTM zoom 11 (approximately 1.185 m/pixel), close to its 1 m source resolution. NAC is optional because its narrow, sparse coverage may leave some selected full tiles without imagery; static context can still be produced for those tiles.

In [ ]:
nac_records = create_tiles_for_query(
    query=TileAOIQuery(**NAC_AOI_BOUNDS),
    output_dir=OUTPUT_DIR / "nac",
    dynamic_sources=((NAC_SOURCE,) if NAC_SOURCE is not None else ()),
    static_sources=((NAC_STATIC_SOURCE,) if NAC_STATIC_SOURCE is not None else ()),
    product_ids=({"nac": NAC_PRODUCT_ID} if INCLUDE_DYNAMIC else None),
    include_dynamic=INCLUDE_DYNAMIC,
    include_static=INCLUDE_STATIC,
    stdout=sys.stdout,
)
print_record_summary(nac_records)
resolved_nac_product_ids = sorted({
    record.product_id
    for record in nac_records
    if record.source_name == "nac" and record.product_id is not None
})
print(f"Resolved NAC product IDs: {resolved_nac_product_ids}")

if nac_records:
    nac_figure = plot_tiling_records(
        nac_records,
        dynamic_source="nac",
        dynamic_label="NAC",
        dynamic_band_number=NAC_BAND_NUMBER,
        static_band_name=STATIC_BAND_TO_PLOT,
        output_path=OUTPUT_DIR / "plots" / "nac_cubes.png",
        max_tiles=MAX_PLOT_TILES,
    )
else:
    nac_figure = None
    print("No NAC or static records were produced; skipping the NAC plot.")

## Alternate queries: geographic point and explicit tile address

When `RUN_ALTERNATE_QUERIES=True`, the first alternate call converts the center of `WAC_AOI_BOUNDS` into a `TilePointQuery`. The automatic workflow needs only lunar latitude and longitude: it routes the point, selects the WAC family zoom, prepares enabled indexes, and applies the same optional-PID and source-mode behavior as the main AOI example. A point query writes the complete 512×512 tile containing that coordinate.

The second call demonstrates the advanced, read-only `create_tiles_for_index()` entry point. It takes the complete `(grid_id, zoom_level, tile_x, tile_y)` address from the first structured point-query record and reuses that record's concrete product ID. Explicit-address tiling does not perform automatic routing, product discovery, or index preparation, so it should be used only with already prepared source definitions and a resolved selector. Separate output directories prevent either alternate query from overwriting the main AOI cubes.

In [ ]:
if RUN_ALTERNATE_QUERIES:
    example_lat = (WAC_AOI_BOUNDS["ul_lat"] + WAC_AOI_BOUNDS["lr_lat"]) / 2
    example_lon = (WAC_AOI_BOUNDS["ul_lon"] + WAC_AOI_BOUNDS["lr_lon"]) / 2

    point_records = create_tiles_for_query(
        query=TilePointQuery(lat=example_lat, lon=example_lon),
        output_dir=OUTPUT_DIR / "wac_point",
        dynamic_sources=((WAC_SOURCE,) if WAC_SOURCE is not None else ()),
        static_sources=((WAC_STATIC_SOURCE,) if WAC_STATIC_SOURCE is not None else ()),
        product_ids=({"wac": WAC_PRODUCT_ID} if INCLUDE_DYNAMIC else None),
        include_dynamic=INCLUDE_DYNAMIC,
        include_static=INCLUDE_STATIC,
        stdout=sys.stdout,
    )
    print("Point-query records:")
    print_record_summary(point_records)
    if point_records:
        point_figure = plot_tiling_records(
            point_records,
            dynamic_source="wac",
            dynamic_label="WAC point query",
            dynamic_band_number=WAC_BAND_NUMBER,
            static_band_name=STATIC_BAND_TO_PLOT,
            output_path=OUTPUT_DIR / "plots" / "wac_point_cubes.png",
            max_tiles=MAX_PLOT_TILES,
        )

        address_record = next(
            (record for record in point_records if record.source_name == "wac"),
            point_records[0],
        )
        explicit_sources = tuple(
            definition.source
            for definition in (WAC_SOURCE, WAC_STATIC_SOURCE)
            if definition is not None
            and (definition.source.name != "wac" or address_record.product_id is not None)
        )
        explicit_selectors = (
            {"wac": address_record.product_id}
            if address_record.source_name == "wac"
            else None
        )
        tile_index_config = TileConfig(
            output_dir=OUTPUT_DIR / "wac_tile_index",
            zoom_level=address_record.zoom_level,
            sources=explicit_sources,
        )
        tile_index_records = create_tiles_for_index(
            tile_index_config,
            grid_id=address_record.grid_id,
            tile_x=address_record.tile_x,
            tile_y=address_record.tile_y,
            selectors=explicit_selectors,
        )
        print("Explicit tile-index-query records:")
        print_record_summary(tile_index_records)
        tile_index_figure = plot_tiling_records(
            tile_index_records,
            dynamic_source="wac",
            dynamic_label="WAC tile-index query",
            dynamic_band_number=WAC_BAND_NUMBER,
            static_band_name=STATIC_BAND_TO_PLOT,
            output_path=OUTPUT_DIR / "plots" / "wac_tile_index_cubes.png",
            max_tiles=MAX_PLOT_TILES,
        )
    else:
        point_figure = None
        tile_index_figure = None
        print("No point-query records were produced; skipping the point and tile-index plots.")
else:
    print("The alternate point and explicit tile-index queries are disabled.")

## North-polar WAC + staged static tiling

`RUN_POLAR_STATIC_QUERY=True` runs this combined example. An AOI entirely at or above the inclusive 82° routing threshold selects `LPS_N` and polar zoom 4 automatically; no grid or zone identifier is supplied. If the query does not intersect the staged polar WAC product, the tiling backend emits a `ProductAOIWarning`, returns no records, and this notebook skips the plot. Selected full tiles can extend south of the routing boundary even though query ownership switches at 82°.

The static source is intentionally custom rather than the canonical 63-band preset. It indexes only the staged aspect, cosine, and elevation VRTs, standardizes their destination NoData to `-32768`, and declares explicit `LPS_N` support. The staging directory must contain copies of those VRTs plus symlinks to the backing TIFF names referenced inside them. Prepare it once from a shell with the commands below; the notebook validates all six paths before index creation. The WAC product ID remains optional: `None` discovers the intersecting product.

```bash
STAGE=/explore/nobackup/people/ajkerr1/lfm_polar_static/north_60m
SOURCE_ROOT=/explore/nobackup/projects/lfm/processed_data/Lunar/Static_final/topo
mkdir -p "$STAGE"
for layer in asp cos elv; do
    cp "$SOURCE_ROOT/new_vrt/lola_kaguya_60mpp.60n.ste.${layer}.LPS_N.vrt" "$STAGE/"
    ln -s "$SOURCE_ROOT/lola_kaguya_60mpp.60n.ste.${layer}.tif" "$STAGE/"
done
```

In [ ]:
if RUN_POLAR_STATIC_QUERY:
    required_polar_static_paths = tuple(
        POLAR_STATIC_DATA_DIR / name
        for vrt_name in POLAR_STATIC_IMAGE_GLOBS
        for name in (
            vrt_name,
            vrt_name.removesuffix(".LPS_N.vrt") + ".tif",
        )
    )
    missing_polar_static_paths = tuple(
        path for path in required_polar_static_paths if not path.is_file()
    )
    if missing_polar_static_paths:
        raise FileNotFoundError(
            "The staged polar-static VRT/TIFF links are incomplete: "
            f"{missing_polar_static_paths}"
        )

    polar_wac_source = make_wac_tile_source(
        data_dir=POLAR_WAC_DATA_DIR,
        index_path=POLAR_WAC_INDEX,
        location_field=LOCATION_FIELD,
        image_glob="*LPS_N.vrt",  # Avoid the backing TIFF and south-polar VRT.
        index_worker_count=INDEX_WORKER_COUNT,
    )
    polar_static_source = TileSourceDefinition(
        source=TileSourceConfig(
            name="static",
            data_dir=POLAR_STATIC_DATA_DIR,
            index_path=POLAR_STATIC_INDEX,
            location_field=LOCATION_FIELD,
            selection_mode="all_intersecting",
            resampling="bilinear",
            output_nodata=-32768,
            required=True,
        ),
        role="static",
        modality="custom",
        image_globs=POLAR_STATIC_IMAGE_GLOBS,
        zoom_overrides={"lps_n": POLAR_STATIC_ZOOM},
        polar_supported=True,
        rebuild_invalid_index=True,
        index_worker_count=INDEX_WORKER_COUNT,
    )
    polar_wac_static_records = create_tiles_for_query(
        query=TileAOIQuery(**POLAR_WAC_AOI_BOUNDS),
        output_dir=OUTPUT_DIR / "polar_wac_static",
        dynamic_sources=(polar_wac_source,),
        static_sources=(polar_static_source,),
        product_ids={"wac": POLAR_WAC_PRODUCT_ID},
        include_dynamic=True,
        include_static=True,
        stdout=sys.stdout,
    )
    print_record_summary(polar_wac_static_records)
    if polar_wac_static_records:
        polar_static_record = next(
            record for record in polar_wac_static_records
            if record.source_name == "static"
        )
        polar_static_band_to_plot = next(
            (name for name in polar_static_record.band_names if "elv" in name.casefold()),
            polar_static_record.band_names[0],
        )
        polar_wac_static_figure = plot_tiling_records(
            polar_wac_static_records,
            dynamic_source="wac",
            dynamic_label="Polar WAC",
            dynamic_band_number=POLAR_WAC_BAND_NUMBER,
            static_band_name=polar_static_band_to_plot,
            output_path=OUTPUT_DIR / "plots" / "polar_wac_static_cubes.png",
            max_tiles=MAX_PLOT_TILES,
        )
    else:
        polar_wac_static_figure = None
        print("No polar WAC/static records were produced; skipping the polar plot.")
else:
    print("The north-polar WAC + static query is configured but disabled.")

## Outputs

The timestamped run directory beneath `notebooks/outputs/tiling/` contains separate WAC, NAC, optional point and explicit-index, and north-polar WAC + staged-static cube directories, plus saved PNG visualizations. Reusable custom source indexes live in `notebooks/outputs/tiling/indexes/`, while canonical shared indexes remain in their data directories and the polar WAC index remains intentionally run-local. Plots convert masked sentinel pixels to `float64` NaN and show at most four tile/product samples. Returned records remain the authoritative machine-readable source, product, grid, zoom, tile, band, CRS, and NoData metadata.

In [ ]:
print(f"Completed tiling notebook run: {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")